[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day04-multi-head-attention-rope.ipynb)
# Day 4 — Multi-Head Attention + RoPE
Split attention into parallel heads, count the parameters exactly, and implement rotary position embeddings with a numeric proof that they encode *relative* position. CPU-only (T4 optional for timing).

In [ ]:
!pip install -q numpy matplotlib torch --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no output

## 1. Multi-head attention from scratch
d_model=512, 8 heads -> each head works on a 64-wide slice. Four projections (Q, K, V, O), no biases.

In [ ]:
import math, torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split(self, w, x):
        B, T, _ = x.shape
        return w(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self._split(self.W_q, x), self._split(self.W_k, x), self._split(self.W_v, x)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        scores = scores.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1), float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

torch.manual_seed(4)
mha = MultiHeadAttention(512, 8)
x = torch.randn(2, 128, 512)
out = mha(x)
assert out.shape == (2, 128, 512), out.shape
print("output shape:", tuple(out.shape))  # Expected: (2, 128, 512)

## 2. Parameter audit
Expect exactly 4 x 512^2 = 1,048,576, i.e. 262,144 per projection.

In [ ]:
total = sum(p.numel() for p in mha.parameters())
per_proj = sum(p.numel() for p in mha.W_q.parameters())
print(f"total params : {total:,}")      # Expected: 1,048,576
print(f"per proj     : {per_proj:,}")    # Expected: 262,144
print(f"4 x 512^2    : {4 * 512 ** 2:,}") # Expected: 1,048,576
assert total == 4 * 512 ** 2 == 1_048_576
print("OK: matches 4 x d_model^2")

## 3. Manual MHA vs the fused SDPA path
Same weights, two code paths. The math is identical; only kernel fusion differs.
(This gap is the case for FlashAttention, Day 47.)

In [ ]:
import time
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)  # Expected: cpu (or cuda on a T4 runtime)
mha = mha.to(device).eval()
T = 2048
xb = torch.randn(1, T, 512, device=device)

def mha_fast(mod, x):  # same weights, fused attention kernel
    B, T, _ = x.shape
    Q = mod._split(mod.W_q, x); K = mod._split(mod.W_k, x); V = mod._split(mod.W_v, x)
    a = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    return mod.W_o(a.transpose(1, 2).contiguous().view(B, T, mod.d_model))

with torch.no_grad():
    ref = mha(xb); fast = mha_fast(mha, xb)
    print("max abs diff:", (ref - fast).abs().max().item())  # Expected: < 1e-4
    assert torch.allclose(ref, fast, atol=1e-4)
    for fn, name in [(mha, "manual per-head"), (lambda z: mha_fast(mha, z), "fused SDPA")]:
        fn(xb)  # warmup
        t0 = time.perf_counter()
        for _ in range(3):
            fn(xb)
        dt = (time.perf_counter() - t0) / 3
        print(f"{name:>14}: {dt*1000:7.1f} ms/forward")
    # Expected on CPU: fused SDPA noticeably faster than the manual loop.

## 4. RoPE: position as rotation
Rotate each (x[2i], x[2i+1]) pair by position x freq. Then (R_m q) . (R_n k) depends only on (m - n).

In [ ]:
def rope_tables(seq_len, freqs):
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)          # (seq, d/2)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    # x: (..., seq, d); cos/sin: (seq, d/2) -- broadcasts over leading dims
    xe, xo = x[..., 0::2], x[..., 1::2]
    re = xe * cos - xo * sin
    ro = xe * sin + xo * cos
    return torch.stack([re, ro], dim=-1).flatten(-2)

def rope_at(x, pos, freqs):
    ang = pos * freqs
    return apply_rope(x, torch.cos(ang)[None, :], torch.sin(ang)[None, :])

# Worked example from the packet: d=2, theta=0.1, q=[1,0] at m=5, k=[1,0] at n=2
freqs = torch.tensor([0.1])
q = torch.tensor([[[1.0, 0.0]]])   # (1, seq=1, d=2)
k = torch.tensor([[[1.0, 0.0]]])
qr = rope_at(q, 5, freqs)
kr = rope_at(k, 2, freqs)
dot = (qr * kr).sum().item()
print("rotated q:", qr.flatten().tolist())  # Expected: [0.878, 0.478]
print("rotated k:", kr.flatten().tolist())  # Expected: [0.980, 0.199]
print(f"dot = {dot:.3f}, cos(0.3) = {math.cos(0.3):.3f}")  # Expected: 0.955, 0.955
assert abs(dot - math.cos(0.3)) < 1e-6
print("OK: dot product = cos(angular distance)")

## 5. The relative-position identity, on random vectors
The whole RoPE claim in one assert: (R_m q) . (R_n k) == (R_{m-n} q) . k, to 1e-5.

In [ ]:
torch.manual_seed(4)
d, base = 64, 10000.0
freqs = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
m, n = 17, 42
q = torch.randn(1, 1, d)
k = torch.randn(1, 1, d)

lhs = (rope_at(q, m, freqs) * rope_at(k, n, freqs)).sum()
rhs = (rope_at(q, m - n, freqs) * k).sum()   # m-n = -25: works for negative positions too
diff = abs(lhs.item() - rhs.item())
print(f"(R_17 q).(R_42 k) = {lhs.item():.6f}")
print(f"(R_-25 q).k       = {rhs.item():.6f}")
print(f"|diff| = {diff:.2e}")                # Expected: < 1e-5
assert diff < 1e-5
print("OK: relative position only -- absolute positions 17 and 42 cancelled out")

## 6. Why long-context models raise the RoPE base
Slowest pair's wavelength = 2 pi / theta_min. Base 10k aliases at ~54k positions; base 500k reaches ~2.5M.

In [ ]:
for base in [10_000, 500_000]:
    freqs = 1.0 / (base ** (torch.arange(0, 128, 2).float() / 128))
    slowest = 2 * math.pi / freqs[-1].item()
    fastest = 2 * math.pi / freqs[0].item()
    print(f"base={base:>7,}: slowest wavelength = {slowest:>10,.0f} positions | fastest = {fastest:.1f}")
# Expected:
# base= 10,000: slowest wavelength =     54,410 positions
# base=500,000: slowest wavelength =  2,559,196 positions

## Wrap-up
- MHA = 4 x d_model^2 params/layer (1,048,576 at width 512). Attention is ~27% of a Llama layer; the MLP is the rest (tomorrow).
- RoPE makes q . k depend on relative distance only -- verified numerically above.
- Tomorrow: the SwiGLU MLP, pre-norm residuals, and the full 8B parameter audit.